In [1]:
import os

# Force single-threaded execution for deterministic results
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import random
import pandas as pd
import numpy as np
import scanpy as sc
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import squidpy as sq
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import matplotlib.colors as mcolors



# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
sc.settings.seed = 42

# === INPUT PATHS ===
input_folder = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_filtered/"
file_name_list_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/file_name_list.csv"
common_mzs_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters.csv"

/opt/anaconda3/lib/python3.12/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [2]:
# Read file names from CSV
file_name_df = pd.read_csv(file_name_list_csv_input_path)
file_names = file_name_df["file_name"].tolist()

In [3]:

# Create mapping from file name → short name
file_name_dict = {}
for fn in file_names:
    parts = fn.split("_")
    group = parts[0][0]         # First letter of AFM/YFM
    condition = parts[1]        # AD or C
    sample_num = parts[2]       # Sample number
    short_name = f"{group}{condition}_{sample_num}"
    file_name_dict[fn] = short_name

In [4]:
# Your list of sample IDs (also used as keys in the dictionary)
aad_1 = sc.read_h5ad(os.path.join(input_folder, "aad_1_filtered.h5ad"))
aad_2 = sc.read_h5ad(os.path.join(input_folder, "aad_2_filtered.h5ad"))
aad_3 = sc.read_h5ad(os.path.join(input_folder, "aad_3_filtered.h5ad"))
aad_4 = sc.read_h5ad(os.path.join(input_folder, "aad_4_filtered.h5ad"))
ac_1 = sc.read_h5ad(os.path.join(input_folder, "ac_1_filtered.h5ad"))
ac_2 = sc.read_h5ad(os.path.join(input_folder, "ac_2_filtered.h5ad"))
ac_3 = sc.read_h5ad(os.path.join(input_folder, "ac_3_filtered.h5ad"))
ac_4 = sc.read_h5ad(os.path.join(input_folder, "ac_4_filtered.h5ad"))
yad_1 = sc.read_h5ad(os.path.join(input_folder, "yad_1_filtered.h5ad"))
yad_2 = sc.read_h5ad(os.path.join(input_folder, "yad_2_filtered.h5ad"))
yad_3 = sc.read_h5ad(os.path.join(input_folder, "yad_3_filtered.h5ad"))
yad_4 = sc.read_h5ad(os.path.join(input_folder, "yad_4_filtered.h5ad"))
yc_1 = sc.read_h5ad(os.path.join(input_folder, "yc_1_filtered.h5ad"))
yc_2 = sc.read_h5ad(os.path.join(input_folder, "yc_2_filtered.h5ad"))
yc_3 = sc.read_h5ad(os.path.join(input_folder, "yc_3_filtered.h5ad"))
yc_4 = sc.read_h5ad(os.path.join(input_folder, "yc_4_filtered.h5ad"))

In [5]:
sample_list = [
    yc_1, yc_2, yc_3, yc_4,
    yad_1, yad_2, yad_3, yad_4,
    ac_1, ac_2, ac_3, ac_4,
    aad_1, aad_2, aad_3, aad_4
]

sample_ids = [
    "yc_1", "yc_2", "yc_3", "yc_4",
    "yad_1", "yad_2", "yad_3", "yad_4",
    "ac_1", "ac_2", "ac_3", "ac_4",
    "aad_1", "aad_2", "aad_3", "aad_4"
]

# Map sample ID → AnnData object
sample_map = dict(zip(sample_ids, sample_list))


In [6]:
for i, sample in enumerate(sample_list):
    # Example: reindex obs to ensure it matches the current data
    sample_list[i].obs = sample.obs.reindex(sample.obs.index)


In [7]:
# Assuming sample_list contains your AnnData objects and their variable names are like 'aad_1', 'ac_1', etc.
# They must be defined variables or in a dictionary for easy iteration.

# Example: if you have them in a dictionary like sample_map = {'aad_1': aad_1, ...}

for sample_id in sample_list:
    adata = sample_id  # Get the AnnData object by variable name string
    # Sum all intensities for each spot (row) across all m/z (columns)
    tic = adata.X.sum(axis=1)  # shape (n_spots, 1) or (n_spots,) depending on sparse or dense
    
    # If sparse, convert to flat array
    if hasattr(tic, "A1"):  # sparse matrix
        tic = tic.A1
    else:
        tic = np.asarray(tic).flatten()

    # Add TIC as a new column in obs
    adata.obs["Processed_TIC"] = tic

In [8]:
# === STEP 3: Load common m/z list ===
common_mz_df = pd.read_csv(common_mzs_csv_input_path)
common_mz_df

,mzs,group,sample,common_group_name
0,326.1952,AC,1,326.1952
1,326.1952,AAD,1,326.1952
2,326.1952,AAD,3,326.1952
3,326.1952,AAD,2,326.1952
4,326.1952,AAD,4,326.1952
...,...,...,...,...
8555,1256.9180,AAD,4,1256.9080
8556,1256.9180,AC,1,1256.9080
8557,1256.9218,AC,3,1256.9080
8558,1256.9218,AC,4,1256.9080


In [9]:

# Example: convert group/sample to sample_id that matches your sample_map keys
def make_sample_id(row):
    # lower group, then underscore, then sample number
    return f"{row['group'].lower()}_{row['sample']}"

common_mz_df['sample_id'] = common_mz_df.apply(make_sample_id, axis=1)
print(common_mz_df)

            mzs group  sample  common_group_name sample_id
0      326.1952    AC       1           326.1952      ac_1
1      326.1952   AAD       1           326.1952     aad_1
2      326.1952   AAD       3           326.1952     aad_3
3      326.1952   AAD       2           326.1952     aad_2
4      326.1952   AAD       4           326.1952     aad_4
...         ...   ...     ...                ...       ...
8555  1256.9180   AAD       4          1256.9080     aad_4
8556  1256.9180    AC       1          1256.9080      ac_1
8557  1256.9218    AC       3          1256.9080      ac_3
8558  1256.9218    AC       4          1256.9080      ac_4
8559  1256.9218    AC       2          1256.9080      ac_2

[8560 rows x 5 columns]


In [10]:
pivot_df = common_mz_df.pivot(index='sample_id', columns='common_group_name', values='mzs')
pivot_df

common_group_name,326.1952,337.1898,339.2275,340.2314,341.2417,351.1685,351.2282,352.2324,353.2430,354.2494,...,1023.6721,1024.6758,1025.6855,1088.8147,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9080
sample_id,,,,,,,,,,,,,,,,,,,,,
aad_1,326.1952,337.1898,339.2275,340.2314,341.2434,351.1685,351.2282,352.2324,353.2430,354.2512,...,1023.6736,1024.6824,1025.6922,1088.8234,1114.8364,1115.8402,1140.7520,1142.7729,1255.9191,1256.9180
aad_2,326.1952,337.1898,339.2275,340.2314,341.2434,351.1685,351.2282,352.2324,353.2430,354.2512,...,1023.6736,1024.6824,1025.6922,1088.8234,1114.8364,1115.8402,1140.7520,1142.7729,1255.9191,1256.9180
aad_3,326.1952,337.1898,339.2275,340.2314,341.2434,351.1685,351.2282,352.2324,353.2430,354.2494,...,1023.6736,1024.6773,1025.6871,1088.8180,1114.8364,1115.8402,1140.7520,1142.7729,1255.9128,1256.9180
aad_4,326.1952,337.1898,339.2275,340.2314,341.2434,351.1685,351.2282,352.2324,353.2430,354.2494,...,1023.6736,1024.6824,1025.6922,1088.8180,1114.8364,1115.8402,1140.7520,1142.7729,1255.9128,1256.9180
ac_1,326.1952,337.1898,339.2275,340.2314,341.2417,351.1685,351.2282,352.2324,353.2430,354.2512,...,1023.6787,1024.6824,1025.6922,1088.8180,1114.8364,1115.8402,1140.7577,1142.7729,1255.9128,1256.9180
ac_2,326.1990,337.1936,339.2313,340.2352,341.2455,351.1723,351.2303,352.2362,353.2468,354.2550,...,1023.6825,1024.6862,1025.6909,1088.8218,1114.8402,1115.8440,1140.7615,1142.7710,1255.9166,1256.9218
ac_3,326.1990,337.1919,339.2313,340.2335,341.2455,351.1723,351.2303,352.2362,353.2468,354.2532,...,1023.6774,1024.6811,1025.6909,1088.8218,1114.8402,1115.8440,1140.7615,1142.7710,1255.9166,1256.9218
ac_4,326.1990,337.1919,339.2313,340.2335,341.2455,351.1723,351.2303,352.2362,353.2468,354.2532,...,1023.6774,1024.6811,1025.6909,1088.8218,1114.8402,1115.8440,1140.7615,1142.7710,1255.9166,1256.9218
yad_1,326.2060,337.1987,339.2380,340.2402,341.2522,351.1787,351.2367,352.2426,353.2532,354.2596,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118


In [11]:
order = [
    'yc_1', 'yc_2', 'yc_3', 'yc_4',
    'yad_1', 'yad_2', 'yad_3', 'yad_4',
    'ac_1', 'ac_2', 'ac_3', 'ac_4',
    'aad_1', 'aad_2', 'aad_3', 'aad_4'
]

# Assuming pivot_df is your pivoted DataFrame with index = sample_id
pivot_df_sorted = pivot_df.loc[pivot_df.index.intersection(order)]
pivot_df_sorted = pivot_df_sorted.loc[order]

pivot_df_sorted

common_group_name,326.1952,337.1898,339.2275,340.2314,341.2417,351.1685,351.2282,352.2324,353.2430,354.2494,...,1023.6721,1024.6758,1025.6855,1088.8147,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9080
sample_id,,,,,,,,,,,,,,,,,,,,,
yc_1,326.2076,337.2004,339.2380,340.2419,341.2539,351.1787,351.2384,352.2426,353.2532,354.2613,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8331,1115.8369,1140.7539,1142.7633,1255.9067,1256.9118
yc_2,326.2038,337.1966,339.2359,340.2381,341.2501,351.1767,351.2346,352.2406,353.2511,354.2575,...,1023.6734,1024.6771,1025.6869,1088.8169,1114.8293,1115.8331,1140.7501,1142.7653,1255.9092,1256.9080
yc_3,326.2055,337.1983,339.2359,340.2398,341.2518,351.1767,351.2364,352.2406,353.2511,354.2593,...,1023.6734,1024.6771,1025.6869,1088.8169,1114.8349,1115.8387,1140.7501,1142.7653,1255.9092,1256.9143
yc_4,326.1995,337.1924,339.2317,340.2339,341.2459,351.1726,351.2305,352.2365,353.2471,354.2535,...,1023.6760,1024.6797,1025.6895,1088.8147,1114.8329,1115.8367,1140.7539,1142.7691,1255.9141,1256.9130
yad_1,326.2060,337.1987,339.2380,340.2402,341.2522,351.1787,351.2367,352.2426,353.2532,354.2596,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118
yad_2,326.2060,337.2004,339.2380,340.2402,341.2522,351.1787,351.2367,352.2426,353.2532,354.2596,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118
yad_3,326.2060,337.2004,339.2380,340.2419,341.2522,351.1787,351.2384,352.2426,353.2532,354.2613,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118
yad_4,326.2060,337.1987,339.2380,340.2402,341.2522,351.1787,351.2367,352.2426,353.2532,354.2596,...,1023.6721,1024.6758,1025.6855,1088.8152,1114.8276,1115.8313,1140.7482,1142.7633,1255.9067,1256.9118
ac_1,326.1952,337.1898,339.2275,340.2314,341.2417,351.1685,351.2282,352.2324,353.2430,354.2512,...,1023.6787,1024.6824,1025.6922,1088.8180,1114.8364,1115.8402,1140.7577,1142.7729,1255.9128,1256.9180


In [12]:
pivot_df_sorted.columns

Index([ 326.1952,  337.1898,  339.2275,  340.2314,  341.2417,  351.1685,
        351.2282,  352.2324,   353.243,  354.2494,
       ...
       1023.6721, 1024.6758, 1025.6855, 1088.8147, 1114.8276, 1115.8313,
       1140.7482, 1142.7633, 1255.9067,  1256.908],
      dtype='float64', name='common_group_name', length=535)

In [13]:
processed_sample_list = []

In [14]:
for i, sample in enumerate(sample_list):
    # Extract m/z values for this sample
    sample_mzs = sample.var['mzs']

    # Build mapping: mz -> common_group_name
    mz_to_group = dict(zip(common_mz_df['mzs'], common_mz_df['common_group_name']))

    # Find intersection
    common_mzs = sample_mzs[sample_mzs.isin(mz_to_group.keys())]

    # Subset AnnData to only these intersecting m/z values
    sample = sample[:, sample.var_names.isin(common_mzs.index)]

    # Keep original m/z values in a column
    sample.var['mzs_drifted'] = sample_mzs.loc[common_mzs.index]

    # Replace with common_group_name for var_names
    sample.var_names = [mz_to_group[mz] for mz in sample.var['mzs_drifted']]

    # Add mzs as a float column
    sample.var['mzs'] = sample.var_names.astype(float)

    # Update the sample in the list
    processed_sample_list.append(sample)

/tmp/ipykernel_696641/2125144445.py:15: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  sample.var['mzs_drifted'] = sample_mzs.loc[common_mzs.index]
/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:835: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    [326.1952, 337.1898, 339.2275, 340.2314, 341.2417]

    Inferred to be: floating

  names = self._prep_dim_index(names, "var")
/tmp/ipykernel_696641/2125144445.py:15: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  sample.var['mzs_drifted'] = sample_mzs.loc[common_mzs.index]
/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:835: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    [326.1952, 337.1898, 339.2275, 340.2314, 341.2417]

    Inferred to be: floating

  names = self._prep_dim_index(names, "var")
/tmp/ipykernel_696

In [15]:
adata = sc.concat(processed_sample_list, label="sample_name", keys = sample_ids)

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [16]:
adata

AnnData object with n_obs × n_vars = 221976 × 535
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'leiden_res_0.02', 'Processed_TIC', 'sample_name'
    obsm: 'spatial'

In [17]:
# Drop from obsm
for key in ['spatial']:
    if key in adata.obsm:
        del adata.obsm[key]

# Drop from obs
if 'leiden_res_0.02' in adata.obs:
    adata.obs = adata.obs.drop(columns=['leiden_res_0.02'])

In [18]:
adata.var["common_mzs"] = adata.var_names.astype(float)

In [19]:
adata.var

,common_mzs
326.1952,326.1952
337.1898,337.1898
339.2275,339.2275
340.2314,340.2314
341.2417,341.2417
...,...
1115.8313,1115.8313
1140.7482,1140.7482
1142.7633,1142.7633
1255.9067,1255.9067


In [20]:
def setup_spatial_data(adata, image_size_cm, image_resolution):
    """
    Sets up a blank spatial dictionary for visualization.
    """
    img_size_px = int(image_size_cm / image_resolution)  # Convert cm to pixels
    blank_image = np.zeros((img_size_px, img_size_px, 3), dtype=np.uint8)

    adata.uns['spatial'] = {
        'spatial': {
            'scalefactors': {
                'tissue_hires_scalef': 1.0,
                'spot_diameter_fullres': 0.05  # arbitrary value
            },
            'images': {
                'hires': blank_image,
                'lowres': blank_image
            }
        }
    }

    if 'x' not in adata.obs or 'y' not in adata.obs:
        raise ValueError("adata.obs must contain 'x' and 'y' coordinates.")

    print("Spatial data setup complete.")
    return adata

In [21]:
adata = setup_spatial_data(adata, image_size_cm =20, image_resolution=0.001)
adata

Spatial data setup complete.


AnnData object with n_obs × n_vars = 221976 × 535
    obs: 'x', 'y', 'TIC', 'sample', 'batch', 'age_group', 'disease_status', 'Processed_TIC', 'sample_name'
    var: 'common_mzs'
    uns: 'spatial'

In [22]:
adata.raw = adata.copy()

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [23]:
adata.obs

,x,y,TIC,sample,batch,age_group,disease_status,Processed_TIC,sample_name
303,129,5,5.347563e+06,1-1,Slide_1,Young,Control,4772702.5,yc_1
304,130,5,5.656035e+06,1-1,Slide_1,Young,Control,5092582.0,yc_1
305,131,5,5.593019e+06,1-1,Slide_1,Young,Control,4998995.0,yc_1
306,132,5,5.479651e+06,1-1,Slide_1,Young,Control,4928391.0,yc_1
307,133,5,5.617997e+06,1-1,Slide_1,Young,Control,5059264.0,yc_1
...,...,...,...,...,...,...,...,...,...
14940,81,102,7.168161e+06,4-4,Slide_4,Aged,AD,6604964.0,aad_4
14941,82,102,7.088457e+06,4-4,Slide_4,Aged,AD,6511722.0,aad_4
14942,83,102,7.158428e+06,4-4,Slide_4,Aged,AD,6631092.0,aad_4
14999,80,103,7.460760e+06,4-4,Slide_4,Aged,AD,6923071.0,aad_4


In [24]:
# preprocess
sc.pp.normalize_total(adata), #target_sum=1e6)
sc.pp.log1p(adata)
sc.pp.scale(adata, max_value=10)

# dimension reduction first
sc.tl.pca(adata, n_comps=50, random_state=42, svd_solver="arpack")

# then neighbors on PCA
# Before running leiden, also fix the neighbors step:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, random_state=42)